# 00 Data check: TNG and Bolshoi

Loads the density grids and T-web masks on Drive, checks that they look right, and recomputes the masks with the `deepvoid` label code to confirm both agree.

Before running: add a Colab secret named `GITHUB_TOKEN` (key icon on the left) holding a GitHub token that can read `skumagai3/deepvoid`, with notebook access switched on. Use a **High-RAM** runtime; no GPU needed.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!pip -q install "git+https://{token}@github.com/skumagai3/deepvoid.git"
del token

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from deepvoid import Catalog
from deepvoid.labels import CLASS_NAMES, tweb_labels
from deepvoid.evaluate import confusion_matrix, scores

cat = Catalog()
print('data root:', cat.root)
for name in cat.names():
    print(f"{name:28s} {'found' if cat.path(name).exists() else 'MISSING'}")

## 1. Load and look

In [ ]:
def fractions(mask):
    c = np.bincount(mask.ravel(), minlength=4) / mask.size
    return {k: round(float(v), 3) for k, v in zip(CLASS_NAMES, c)}

data = {}
for sim in ('tng300-3', 'bolshoi'):
    dm, mask = cat.load(f'{sim}/dm'), cat.load(f'{sim}/mask')
    data[sim] = (dm, mask)
    print(sim, '| n =', dm.n, '| cell =', round(dm.cell_size, 3), 'Mpc/h',
          '| density mean =', f'{dm.data.mean():.3g}', '| mask classes:', fractions(mask.data))

In [ ]:
cmap = plt.matplotlib.colors.ListedColormap(['#2b6cb0', '#9ae6b4', '#f6ad55', '#c53030'])
fig, ax = plt.subplots(2, 2, figsize=(12, 12))
for row, (sim, (dm, mask)) in enumerate(data.items()):
    k = dm.n // 2
    ax[row, 0].imshow(np.log10(dm.data[k] / dm.data.mean() + 1e-3).T, origin='lower', cmap='gray')
    ax[row, 1].imshow(mask.data[k].T, origin='lower', cmap=cmap, vmin=0, vmax=3, interpolation='nearest')
    ax[row, 0].set_title(f'{sim} log density'); ax[row, 1].set_title(f'{sim} mask')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 2. Recompute the masks with the new code

Uses each simulation's original smoothing (TNG: 2.4 cells; Bolshoi: 0.916 Mpc/h) and threshold 0.65, then compares voxel by voxel. Agreement near 100% means the label code and the files match. Takes a few minutes per box.

In [ ]:
original_sigma_cells = {
    'tng300-3': 2.4,
    'bolshoi': 0.916 / (250 / 640),
}
for sim, (dm, mask) in data.items():
    new = tweb_labels(dm, threshold=0.65, sigma_cells=original_sigma_cells[sim])
    agree = (new.data == mask.data).mean()
    s = scores(confusion_matrix(mask.data, new.data))
    print(f"{sim}: {agree:.2%} of voxels agree | MCC {s['mcc']:.3f} | new fractions {fractions(new.data)}")

If Bolshoi agrees much less than TNG, the stored Bolshoi mask was probably made differently (its file name mentions a 416 grid). Paste the printed numbers back to Claude either way.